# 05 · Open-set recognition & calibration
A segment is **accepted** only if its top-1 score/margin pass thresholds calibrated for top-1 precision; **uncertain** if the correct word is likely in the top-5; otherwise **unknown** — reported with the nearest known word instead of a guess.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'modules').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['Tahoma', 'Leelawadee UI', 'DejaVu Sans']
from IPython.display import display, Image, Audio, Markdown
from modules.utils import ART, read_json
REP, V4, MAN4 = ART / 'reports', ART / 'v4', ART / 'manifests_v4'
pd.set_option('display.max_colwidth', 80)

In [2]:
cal = read_json(REP/'v4_calibration.json'); cal

{'accept': {'tau_score': 0.6384773254394531, 'tau_margin': 0.0710340142250061},
 'uncertain': {'tau_score': 0.5644997358322144, 'tau_margin': 0.0},
 'rule': 'thresholds = quantiles of val in-vocab CORRECT matches: likely ≥ median score & median margin; unknown < 5th pct score',
 'empirical': {'val': {'likely': {'share': 0.22641509433962265,
    'n': 36,
    'top1_precision_in_vocab': 0.2857142857142857,
    'top5_hit_in_vocab': 0.35714285714285715,
    'oov_share': 0.6111111111111112},
   'candidate': {'share': 0.6666666666666666,
    'n': 106,
    'top1_precision_in_vocab': 0.09523809523809523,
    'top5_hit_in_vocab': 0.2619047619047619,
    'oov_share': 0.6037735849056604},
   'unknown': {'share': 0.1069182389937107,
    'n': 17,
    'top1_precision_in_vocab': 0.16666666666666666,
    'top5_hit_in_vocab': 0.3333333333333333,
    'oov_share': 0.6470588235294118}},
  'test': {'likely': {'share': 0.16774193548387098,
    'n': 26,
    'top1_precision_in_vocab': 0.3,
    'top5_hit_in_voc

Spotted segments use the same score/margin rule. Measured on held-out-signer synthetic sentences (bank without that signer) — the tiers are only weakly informative:

In [3]:
print(open(REP/'v4_spot_tier_check.txt', encoding='utf-8').read())
print(open(REP/'v4_spot_agg_check.txt', encoding='utf-8').read())

tune n 622 in-vocab 124 top1 acc in-vocab 0.081 top5 0.258
  E1-calib accepted share 0.166 in-vocab prec 0.091 top5 0.364 n_inv 22 oov share 0.79
  E1-calib uncertain share 0.617 in-vocab prec 0.068 top5 0.227 n_inv 88 oov share 0.77
  E1-calib unknown share 0.217 in-vocab prec 0.143 top5 0.286 n_inv 14 oov share 0.9
  margin>=q0.5 in-vocab prec 0.101 n_inv 69
  margin>=q0.7 in-vocab prec 0.095 n_inv 42
  margin>=q0.9 in-vocab prec 0.154 n_inv 13
test n 609 in-vocab 134 top1 acc in-vocab 0.112 top5 0.224
  E1-calib accepted share 0.194 in-vocab prec 0.138 top5 0.345 n_inv 29 oov share 0.75
  E1-calib uncertain share 0.589 in-vocab prec 0.103 top5 0.207 n_inv 87 oov share 0.76
  E1-calib unknown share 0.217 in-vocab prec 0.111 top5 0.111 n_inv 18 oov share 0.86
  margin>=q0.5 in-vocab prec 0.1 n_inv 70
  margin>=q0.7 in-vocab prec 0.136 n_inv 44
  margin>=q0.9 in-vocab prec 0.125 n_inv 16

tune n in-vocab 108 {'single': (0.093, 0.296), 'vote': (0.093, 0.333), 'sum': (0.102, 0.352), 'max